This notebook ingests and prepares data from the GP Patient Survey (GPPS) 2025 ICS-level dataset, a national survey capturing patient experience of primary care services across England. The purpose is to create a structured, analysis-ready dataset that reflects variation in access, experience, and confidence in GP services, alongside the ethnic composition of local populations. This forms a key local system layer to complement national health outcome data.

In [1]:
# Inspect GPPS CSV structure

import pandas as pd
from pathlib import Path

GPPS_PATH = Path("GPPS_2025_ICS_data_(weighted)_(csv)_v2_PUBLIC_v2.csv")

if not GPPS_PATH.exists():
    raise FileNotFoundError(f"{GPPS_PATH} not found in project folder")

# Read a preview
gpps_preview = pd.read_csv(GPPS_PATH, nrows=50)

print("Shape of preview:", gpps_preview.shape)
print("\nColumns:")
print(gpps_preview.columns.tolist())

# Simple structural scan
cols = gpps_preview.columns.tolist()
cols_lower = [str(c).lower() for c in cols]

summary = {
    "rows_preview": gpps_preview.shape[0],
    "cols": len(cols),
    "has_ethnicity": any("ethnic" in c for c in cols_lower),
    "has_black": any("black" in c for c in cols_lower),
    "has_access_terms": any(x in " ".join(cols_lower) for x in ["appointment", "access", "waiting"]),
    "has_experience_terms": any(x in " ".join(cols_lower) for x in ["experience", "confidence", "trust"]),
    "columns_preview_first_20": cols[:20],
}

print("\nSummary:")
for k, v in summary.items():
    print(f"{k}: {v}")

display(gpps_preview.head(10))

Shape of preview: (42, 1239)

Columns:
['ad_icscode', 'ad_icsname', 'ad_icscodeons', 'ad_commissioningregioncode', 'ad_commissioningregionname', 'distributed', 'received', 'resprate', 'localgpservicesphone_1.count', 'localgpservicesphone_2.count', 'localgpservicesphone_3.count', 'localgpservicesphone_4.count', 'localgpservicesphone_5.count', 'localgpservicesphone_6.count', 'localgpservicesphone.counteval', 'localgpservicesphone.basefulluw', 'localgpservicesphone.baseevaluw', 'localgpservicesphone.basefullw', 'localgpservicesphone.baseevalw', 'localgpservicesphone_1.pct', 'localgpservicesphone_2.pct', 'localgpservicesphone_3.pct', 'localgpservicesphone_4.pct', 'localgpservicesphone_5.pct', 'localgpservicesphone_6.pct', 'localgpservicesphone.pcteval', 'localgpservicesphone.lowileval', 'localgpservicesphone.hiwileval', 'localgpserviceswebsite_1.count', 'localgpserviceswebsite_2.count', 'localgpserviceswebsite_3.count', 'localgpserviceswebsite_4.count', 'localgpserviceswebsite_5.count', 'l

,ad_icscode,ad_icsname,ad_icscodeons,ad_commissioningregioncode,ad_commissioningregionname,distributed,received,resprate,localgpservicesphone_1.count,localgpservicesphone_2.count,...,aboutyouagemerged_1.pct,aboutyouagemerged_2.pct,aboutyouagemerged_3.pct,aboutyouagemerged_4.pct,aboutyouagemerged_5.pct,aboutyouagemerged_6.pct,aboutyouagemerged_7.pct,aboutyouagemerged_8.pct,aboutyouagemerged_9.pct,popsize
0,QE1,LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CARE S...,E54000048,Y62,NORTH WEST COMMISSIONING REGION,83553,22201,0.265712,1250.818144,4526.414049,...,0.089787,0.145935,0.157752,0.158968,0.178441,0.136070,0.096349,0.028998,0.007700,1502967
1,QF7,SOUTH YORKSHIRE INTEGRATED CARE SYSTEM,E54000061,Y63,NORTH EAST AND YORKSHIRE COMMISSIONING REGION,70567,18703,0.265039,1125.081875,2811.620562,...,0.098495,0.165644,0.174803,0.160020,0.166693,0.121593,0.082739,0.023936,0.006077,1221990
2,QGH,HEREFORDSHIRE AND WORCESTERSHIRE INTEGRATED CA...,E54000019,Y60,MIDLANDS COMMISSIONING REGION,24593,9070,0.368804,626.020930,2275.102337,...,0.089196,0.127186,0.146025,0.154560,0.184062,0.144799,0.110630,0.036996,0.006546,682549
3,QH8,MID AND SOUTH ESSEX INTEGRATED CARE SYSTEM,E54000026,Y61,EAST OF ENGLAND COMMISSIONING REGION,55368,16038,0.289662,715.075600,2193.289967,...,0.076576,0.153411,0.171640,0.174974,0.169983,0.124562,0.094848,0.028110,0.005897,1022635
4,QHG,"BEDFORDSHIRE, LUTON AND MILTON KEYNES INTEGRAT...",E54000024,Y61,EAST OF ENGLAND COMMISSIONING REGION,35703,9626,0.269613,869.910797,1558.734214,...,0.079318,0.172599,0.206950,0.180962,0.158356,0.104166,0.068299,0.021696,0.007654,865117
5,QHL,BIRMINGHAM AND SOLIHULL INTEGRATED CARE SYSTEM,E54000055,Y60,MIDLANDS COMMISSIONING REGION,116148,20438,0.175965,990.667945,2903.216693,...,0.112847,0.183969,0.195873,0.169775,0.146850,0.096379,0.059833,0.022150,0.012325,1264860
6,QHM,NORTH EAST AND NORTH CUMBRIA INTEGRATED CARE S...,E54000050,Y63,NORTH EAST AND YORKSHIRE COMMISSIONING REGION,127531,38587,0.302570,2080.471801,8129.416830,...,0.085938,0.152418,0.163046,0.153691,0.177891,0.141541,0.091004,0.028700,0.005772,2620671
7,QJ2,DERBY AND DERBYSHIRE INTEGRATED CARE SYSTEM,E54000058,Y60,MIDLANDS COMMISSIONING REGION,37468,12689,0.338662,828.396305,2037.471429,...,0.080205,0.151624,0.162365,0.163712,0.180816,0.130042,0.096688,0.028851,0.005695,922243
8,QJG,SUFFOLK AND NORTH EAST ESSEX INTEGRATED CARE S...,E54000023,Y61,EAST OF ENGLAND COMMISSIONING REGION,29677,10624,0.357988,936.915934,2171.100562,...,0.083159,0.143335,0.152943,0.157958,0.174877,0.137815,0.109478,0.036453,-98.000000,874681
9,QJK,DEVON INTEGRATED CARE SYSTEM,E54000037,Y58,SOUTH WEST COMMISSIONING REGION,37760,13520,0.358051,1094.476530,3048.267585,...,0.088863,0.127837,0.144552,0.151388,0.182016,0.148662,0.115996,0.036562,-98.000000,1066439


This document is an aggregated survey output, not raw individual-level data.

Key facts:

Only 42 rows → ICS-level dataset (not borough, not patient-level)

1239 columns → each question × response option × metric (count/pct/etc.)

Ethnicity is present, but:

not as clean categories like ONS tables

encoded as:

aboutyouethnicity_*

dv_ethnicityband_*

No explicit “Black” column → ethnicity is coded numerically, not labelled

What this means for the project

Directly answers:

access (appointments, contact methods)

experience (confidence, satisfaction, listening)

Includes ethnicity breakdown → critical for inequality focus

Limitations:

Geography = ICS only → no borough-level insight

Structure = wide + encoded → needs reshaping

Ethnicity = coded bands → must map carefully

Not easily linkable row-by-row with ONS data (different structure + level)

In [53]:
# Load + basic validation
import pandas as pd
from pathlib import Path

GPPS_PATH = Path("GPPS_2025_ICS_data_(weighted)_(csv)_v2_PUBLIC_v2.csv")

if not GPPS_PATH.exists():
    raise FileNotFoundError(f"{GPPS_PATH} not found")

gpps = pd.read_csv(GPPS_PATH, low_memory=False)

print("Shape:", gpps.shape)
print("Columns:", len(gpps.columns))

Shape: (42, 1239)
Columns: 1239


In [ ]:
# Core geography
geo_cols = [
    "ad_icscode",
    "ad_icsname"
]

# Ethnicity bands (counts + pct)
eth_cols = [c for c in gpps.columns if "dv_ethnicityband" in c and ".pct" in c]

# Selected indicators (minimal set)
indicator_patterns = [
    "gpcontactoverall",   # overall experience contacting GP
    "overallexp",         # overall experience
    "healthconfidence"    # confidence managing health
]

indicator_cols = [
    c for c in gpps.columns
    if any(p in c for p in indicator_patterns) and ".pct" in c
]

selected_cols = geo_cols + eth_cols + indicator_cols

gpps_subset = gpps[selected_cols].copy()

print("Selected columns:", len(selected_cols))

Selected columns: 26


In [55]:
gpps_long = gpps_subset.melt(
    id_vars=geo_cols,
    var_name="measure_raw",
    value_name="value"
)

gpps_long = gpps_long.dropna(subset=["value"])

In [57]:
print(gpps_long.columns.tolist())

['ad_icscode', 'ad_icsname', 'measure_raw', 'value', 'metric', 'category', 'measure_type']


In [58]:
def parse_measure(measure):
    if isinstance(measure, str) and measure.startswith("dv_ethnicityband"):
        category = measure.split("_")[-1].replace(".pct", "")
        return "ethnicity_distribution", "ethnicity", category

    parts = str(measure).split("_")
    base = parts[0]
    category = parts[-1].replace(".pct", "")
    
    return base, "response", category


parsed = gpps_long["measure_raw"].apply(parse_measure)

gpps_long["metric"] = parsed.apply(lambda x: x[0])
gpps_long["category"] = parsed.apply(lambda x: x[2])
gpps_long["measure_type"] = parsed.apply(lambda x: x[1])

print(gpps_long["metric"].value_counts().head(10))

metric
ethnicity_distribution      252
gpcontactoverall            210
overallexp                  210
healthconfidence            210
gpcontactoverall.pcteval     42
overallexp.pcteval           42
healthconfidence.pcteval     42
Name: count, dtype: int64


In [59]:
# Extract ethnicity dataset

ethnicity_dist = gpps_long[
    gpps_long["measure_raw"].str.match(r"^dv_ethnicityband_\d+\.pct$", na=False)
].copy()

print("Ethnicity distribution shape:", ethnicity_dist.shape)
print(sorted(ethnicity_dist["measure_raw"].unique()))
display(ethnicity_dist.head())

Ethnicity distribution shape: (252, 7)
['dv_ethnicityband_1.pct', 'dv_ethnicityband_2.pct', 'dv_ethnicityband_3.pct', 'dv_ethnicityband_4.pct', 'dv_ethnicityband_5.pct', 'dv_ethnicityband_6.pct']


,ad_icscode,ad_icsname,measure_raw,value,metric,category,measure_type
0,QE1,LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CARE S...,dv_ethnicityband_1.pct,0.845575,ethnicity_distribution,1,ethnicity
1,QF7,SOUTH YORKSHIRE INTEGRATED CARE SYSTEM,dv_ethnicityband_1.pct,0.834100,ethnicity_distribution,1,ethnicity
2,QGH,HEREFORDSHIRE AND WORCESTERSHIRE INTEGRATED CA...,dv_ethnicityband_1.pct,0.925890,ethnicity_distribution,1,ethnicity
3,QH8,MID AND SOUTH ESSEX INTEGRATED CARE SYSTEM,dv_ethnicityband_1.pct,0.853095,ethnicity_distribution,1,ethnicity
4,QHG,"BEDFORDSHIRE, LUTON AND MILTON KEYNES INTEGRAT...",dv_ethnicityband_1.pct,0.678122,ethnicity_distribution,1,ethnicity


In [68]:
ethnicity_dist["value"] = ethnicity_dist["value"].replace({
    -98: None,
    -99: None
})

Some ethnicity proportions are missing due to suppression in GPPS data (values coded as -98), and totals may not sum to 1.

In [ ]:
# Build experience dataset
gpps_exp = gpps_long[
    gpps_long["metric"].isin([
        "gpcontactoverall",
        "overallexp",
        "healthconfidence"
    ])
].copy()

gpps_exp = gpps_exp[
    ~gpps_exp["measure_raw"].str.contains("pcteval", na=False)
].copy()

gpps_exp = gpps_exp.rename(columns={
    "ad_icscode": "area_code",
    "ad_icsname": "area_name"
})

gpps_exp["area_geography"] = "ICS"
gpps_exp["source"] = "GP Patient Survey"
gpps_exp["year"] = 2025
gpps_exp["geography"] = "England"
gpps_exp["table"] = "GPPS_ICS"

gpps_final = gpps_exp[[
    "table",
    "source",
    "year",
    "geography",
    "area_geography",
    "area_code",
    "area_name",
    "metric",
    "category",
    "measure_raw",
    "value"
]].copy()

print("gpps_final shape:", gpps_final.shape)
display(gpps_final.head())

gpps_final shape: (630, 11)


,table,source,year,geography,area_geography,area_code,area_name,metric,category,measure_raw,value
252,GPPS_ICS,GP Patient Survey,2025,England,ICS,QE1,LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CARE S...,gpcontactoverall,1,gpcontactoverall_1.pct,0.425938
253,GPPS_ICS,GP Patient Survey,2025,England,ICS,QF7,SOUTH YORKSHIRE INTEGRATED CARE SYSTEM,gpcontactoverall,1,gpcontactoverall_1.pct,0.380120
254,GPPS_ICS,GP Patient Survey,2025,England,ICS,QGH,HEREFORDSHIRE AND WORCESTERSHIRE INTEGRATED CA...,gpcontactoverall,1,gpcontactoverall_1.pct,0.448645
255,GPPS_ICS,GP Patient Survey,2025,England,ICS,QH8,MID AND SOUTH ESSEX INTEGRATED CARE SYSTEM,gpcontactoverall,1,gpcontactoverall_1.pct,0.354012
256,GPPS_ICS,GP Patient Survey,2025,England,ICS,QHG,"BEDFORDSHIRE, LUTON AND MILTON KEYNES INTEGRAT...",gpcontactoverall,1,gpcontactoverall_1.pct,0.294084


In [ ]:
# Clean ethnicity dataset
ethnicity_dist = ethnicity_dist.rename(columns={
    "category": "ethnicity_code",
    "ad_icscode": "area_code",
    "ad_icsname": "area_name"
})

ethnicity_dist["metric"] = "population_distribution"

ethnicity_dist["area_geography"] = "ICS"
ethnicity_dist["source"] = "GP Patient Survey"
ethnicity_dist["year"] = 2025
ethnicity_dist["geography"] = "England"
ethnicity_dist["table"] = "GPPS_ICS"

In [ ]:
# Map ethnicity
ethnicity_map = {
    "1": "White",
    "2": "Mixed",
    "3": "Asian",
    "4": "Black",
    "5": "Other",
    "6": "Not stated"
}

ethnicity_dist["ethnicity_standard"] = ethnicity_dist["ethnicity_code"].map(ethnicity_map)
ethnicity_dist["ethnicity_raw"] = ethnicity_dist["ethnicity_code"]

In [ ]:
# Create final ethnicity dataset
ethnicity_dist_final = ethnicity_dist[[
    "table",
    "source",
    "year",
    "geography",
    "area_geography",
    "area_code",
    "area_name",
    "metric",
    "ethnicity_raw",
    "ethnicity_standard",
    "measure_raw",
    "value"
]].copy()

print("ethnicity_dist_final shape:", ethnicity_dist_final.shape)
display(ethnicity_dist_final.head())


ethnicity_dist_final shape: (252, 12)


,table,source,year,geography,area_geography,area_code,area_name,metric,ethnicity_raw,ethnicity_standard,measure_raw,value
0,GPPS_ICS,GP Patient Survey,2025,England,ICS,QE1,LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CARE S...,population_distribution,1,White,dv_ethnicityband_1.pct,0.845575
1,GPPS_ICS,GP Patient Survey,2025,England,ICS,QF7,SOUTH YORKSHIRE INTEGRATED CARE SYSTEM,population_distribution,1,White,dv_ethnicityband_1.pct,0.8341
2,GPPS_ICS,GP Patient Survey,2025,England,ICS,QGH,HEREFORDSHIRE AND WORCESTERSHIRE INTEGRATED CA...,population_distribution,1,White,dv_ethnicityband_1.pct,0.92589
3,GPPS_ICS,GP Patient Survey,2025,England,ICS,QH8,MID AND SOUTH ESSEX INTEGRATED CARE SYSTEM,population_distribution,1,White,dv_ethnicityband_1.pct,0.853095
4,GPPS_ICS,GP Patient Survey,2025,England,ICS,QHG,"BEDFORDSHIRE, LUTON AND MILTON KEYNES INTEGRAT...",population_distribution,1,White,dv_ethnicityband_1.pct,0.678122


In [ ]:
# Final sanity check after suppression handling
ethnicity_sum_check = (
    ethnicity_dist_final
    .groupby("area_code")["value"]
    .sum(min_count=1)
    .reset_index()
)

print(ethnicity_sum_check["value"].describe())
display(ethnicity_sum_check.sort_values("value").head(10))

count     42.0
unique    30.0
top        1.0
freq       3.0
Name: value, dtype: float64


,area_code,value
29,QSL,0.990682
31,QT6,0.992232
10,QJM,0.995007
9,QJK,0.995326
17,QMM,0.995584
2,QGH,0.996791
34,QUE,1.0
13,QKS,1.0
37,QWE,1.0
40,QXU,1.0


In [82]:
# Save outputs
gpps_final.to_parquet("_processed_gpps_survey.parquet", index=False)
ethnicity_dist_final.to_parquet("_processed_gpps_ethnicity_distribution.parquet", index=False)

print("Saved:")
print("_processed_gpps_survey.parquet")
print("_processed_gpps_ethnicity_distribution.parquet")


Saved:
_processed_gpps_survey.parquet
_processed_gpps_ethnicity_distribution.parquet


GPPS ethnicity distribution may contain suppressed values coded in source data and converted to missing.

In [ ]:
# Sanity check 
print(gpps_final.shape)
print(gpps_final["metric"].value_counts())
print(gpps_final["value"].describe())

(630, 11)
metric
gpcontactoverall    210
overallexp          210
healthconfidence    210
Name: count, dtype: int64
count    630.000000
mean       0.202459
std        0.156314
min        0.023512
25%        0.067332
50%        0.142512
75%        0.313594
max        0.557693
Name: value, dtype: float64


In [84]:
gpps_final.groupby(["area_code", "metric"])["value"].sum().describe()

count    126.000000
mean       1.012294
std        0.018096
min        1.000000
25%        1.000000
50%        1.000000
75%        1.032230
max        1.057220
Name: value, dtype: float64

In [85]:
print(ethnicity_dist_final.shape)
print(ethnicity_dist_final["ethnicity_standard"].value_counts())
print(ethnicity_dist_final["value"].describe())

(252, 12)
ethnicity_standard
White         42
Mixed         42
Asian         42
Black         42
Other         42
Not stated    42
Name: count, dtype: int64
count     244.000000
unique    244.000000
top         0.845575
freq        1.000000
Name: value, dtype: float64


In [89]:
ethnicity_dist_final["value"] = pd.to_numeric(
    ethnicity_dist_final["value"],
    errors="coerce"
)

In [90]:
ethnicity_dist_final.groupby("area_code")["value"].sum(min_count=1).describe()

count    42.000000
mean      0.999181
std       0.002184
min       0.990682
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: value, dtype: float64

The GPPS data has been cleaned, standardised, and reshaped into two aligned datasets: one capturing service experience metrics and another representing ethnicity distribution at ICS level. Data quality issues, including suppressed values and non-comparable metrics, have been addressed, and integrity checks confirm consistency. The outputs are now ready for integration with other sources in subsequent analysis.